# Qwen3.5-0.8B × TinyCeNN — Integrated Memory V2.2

V2.2 keeps the corrected Qwen3.5 cache-position handling, the robust V2.1 cache-equivalence diagnostic, and adds an **interactive chat comparison** between untouched Qwen3.5 and the validation-selected TinyCeNN model.


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

REPO = Path(tempfile.mkdtemp(prefix="qwen35-cenn-v22-")) / "TinyCeNN-LM"
subprocess.run(["git","clone","--quiet","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO)], check=True)
subprocess.run(["git","fetch","origin","main"], cwd=REPO, check=True)
subprocess.run(["git","reset","--hard","origin/main"], cwd=REPO, check=True)

subprocess.run([sys.executable,"-m","pip","install","-q",
                "transformers==5.17.0","huggingface_hub>=0.36.2","datasets>=3,<5",
                "pytest","pandas","matplotlib"], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO),"--no-deps"], check=True)

os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO/"src")])
sys.path[:0] = [str(REPO), str(REPO/"src")]

import torch
from transformers import AutoConfig
MODEL_ID = "Qwen/Qwen3.5-0.8B"
cfg = AutoConfig.from_pretrained(MODEL_ID).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t == "full_attention"]
SOURCE = subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
print("Source:", SOURCE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Full-attention layers:", FULL)
assert cfg.model_type == "qwen3_5_text"
assert FULL == [3,7,11,15,19,23]


## Configuration


In [ ]:
PROFILE = "balanced" # @param ["smoke","balanced","extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}

PROFILES = {
  "smoke": dict(train_contexts="64,128",test_contexts="64,128,256",block_size=16,features=32,train_documents=4,validation_documents=2,test_documents=2,warm_documents=2,warm_steps=2,joint_steps=4,eval_every=2,timing_documents=1,timing_repeats=1,decode_tokens=8,loss_chunk=4),
  "balanced": dict(train_contexts="128,256,512",test_contexts="128,256,512,1024,2048",block_size=32,features=64,train_documents=48,validation_documents=8,test_documents=16,warm_documents=6,warm_steps=40,joint_steps=120,eval_every=20,timing_documents=2,timing_repeats=2,decode_tokens=24,loss_chunk=8),
  "extended": dict(train_contexts="256,512,1024",test_contexts="256,512,1024,2048,4096",block_size=32,features=96,train_documents=96,validation_documents=16,test_documents=32,warm_documents=12,warm_steps=80,joint_steps=240,eval_every=40,timing_documents=3,timing_repeats=3,decode_tokens=32,loss_chunk=8),
}
if not torch.cuda.is_available() and PROFILE != "smoke": raise RuntimeError("Select a GPU runtime or use smoke.")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TinyCeNN/qwen35-integrated-v2-2")
else:
    BASE = Path("/content/qwen35-integrated-v2-2")
BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = BASE / run_id
LOG = BASE / (run_id + ".log")
RUN = dict(PROFILES[PROFILE], seed=2030)
print(json.dumps(RUN, indent=2)); print("Results:", OUT)


## Preflight


In [ ]:
subprocess.run(["git","fetch","origin","main"], cwd=REPO, check=True)
subprocess.run(["git","reset","--hard","origin/main"], cwd=REPO, check=True)
SOURCE = subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
print("Testing source:", SOURCE)
env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
r = subprocess.run([sys.executable,"-m","pytest","-q","tests/test_qwen35_integrated_memory.py"],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f"Qwen3.5 preflight failed: {r.returncode}")
BENCHMARK = REPO / "scripts/benchmark_qwen35_integrated_memory_v2.py"
probe = subprocess.run([sys.executable,str(BENCHMARK),"--help"],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
if probe.returncode: raise RuntimeError("Qwen3.5 V2.1 benchmark entrypoint failed")
print("✅ preflight passed")


## Train + evaluate


In [ ]:
cmd = [sys.executable,"-u",str(BENCHMARK),"--base-model",MODEL_ID,"--output-dir",str(OUT)]
for k,v in RUN.items(): cmd += ["--" + k.replace("_","-"), str(v)]
print("Running:", BENCHMARK.name); print(" ".join(cmd))
try:
    with LOG.open("w") as log:
        with subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end="",flush=True); log.write(line); log.flush()
            status = p.wait()
    if status: raise RuntimeError(f"Run failed: {status}; inspect {LOG}")
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG,OUT/"console.log")
        print("Archive:",shutil.make_archive(str(OUT)+"-results","zip",root_dir=OUT))


## Results


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
s = pd.read_csv(OUT/"integrated_summary.csv")
selected = json.loads((OUT/"selection.json").read_text())["selected"]
print("Locked validation selection:", selected)
display(s.sort_values(["candidate","context"]).reset_index(drop=True))
x = s[s.candidate == selected].sort_values("context")
fig,ax = plt.subplots(figsize=(9,4)); ax.plot(x.context,x.ppl_ratio,marker="o",label="PPL ratio"); ax.plot(x.context,x.total_cache_ratio,marker="s",label="cache ratio"); ax.axhline(1,linestyle="--"); ax.set_xscale("log",base=2); ax.set_xlabel("Context"); ax.legend(); ax.set_title(selected); plt.show()


## Chat comparison — Original Qwen3.5 vs selected TinyCeNN

Edit `USER_MESSAGE` and rerun only this cell. Both models receive the **same chat template** and use deterministic greedy decoding.


In [ ]:
USER_MESSAGE = "Explain in simple terms why the sky is blue." # @param {type:"string"}
SYSTEM_MESSAGE = "You are a helpful assistant." # @param {type:"string"}
MAX_NEW_TOKENS = 128 # @param {type:"integer"}

import gc, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.qwen35_integrated_memory import restore_student, greedy_generate, inference_mode, native_dtype
selection = json.loads((OUT/"selection.json").read_text())
report = json.loads((OUT/"integrated_report.json").read_text())
manifest = json.loads((OUT/"manifest.json").read_text())
selected = selection["selected"]
record = next(r for r in report["candidates"] if r["candidate"] == selected)
checkpoint = OUT / record["checkpoint"]
base_revision = manifest["model_revision"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = native_dtype(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=base_revision)
messages = []
if SYSTEM_MESSAGE.strip(): messages.append({"role":"system","content":SYSTEM_MESSAGE.strip()})
messages.append({"role":"user","content":USER_MESSAGE})
try:
    prompt_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", enable_thinking=False)
except TypeError:
    prompt_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
prompt_ids = prompt_ids.to(device)
original = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=base_revision, dtype=dtype, attn_implementation="sdpa").to(device).eval()
payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
cenn = restore_student(original, payload).to(device).eval()
def _stop_ids(tok):
    ids=[]; eos=tok.eos_token_id
    if eos is not None: ids.extend(eos if isinstance(eos,(list,tuple,set)) else [eos])
    for name in ("<|im_end|>","<|endoftext|>","<|end_of_turn|>"):
        tid=tok.convert_tokens_to_ids(name)
        if isinstance(tid,int) and tid>=0 and tid!=tok.unk_token_id: ids.append(tid)
    return sorted(set(ids))
STOP_IDS=_stop_ids(tokenizer)
with torch.no_grad(): original_tokens,_=greedy_generate(original,prompt_ids,tokens=MAX_NEW_TOKENS,stop_token_ids=STOP_IDS)
with inference_mode(cenn,"float32"): cenn_tokens,_=greedy_generate(cenn,prompt_ids,tokens=MAX_NEW_TOKENS,stop_token_ids=STOP_IDS)
original_text=tokenizer.decode(original_tokens[0],skip_special_tokens=True).strip()
cenn_text=tokenizer.decode(cenn_tokens[0],skip_special_tokens=True).strip()
n=min(original_tokens.shape[1],cenn_tokens.shape[1]); same=(original_tokens[:,:n]==cenn_tokens[:,:n])[0]; agreement=float(same.float().mean()) if n else 0.0; different=(~same).nonzero(as_tuple=False); first_difference=int(different[0].item()) if len(different) else None; exact=bool(original_tokens.shape==cenn_tokens.shape and torch.equal(original_tokens,cenn_tokens))
print("="*100); print("USER:\n"+USER_MESSAGE); print("\n"+"="*100); print("ORIGINAL QWEN3.5:\n"+original_text); print("\n"+"="*100); print(f"TINYCeNN ({selected}, layers={record['layers']}):\n{cenn_text}"); print("\n"+"="*100); print(f"Prompt tokens: {prompt_ids.shape[1]}"); print(f"Original new tokens: {original_tokens.shape[1]}"); print(f"CeNN new tokens: {cenn_tokens.shape[1]}"); print(f"Token agreement: {agreement:.2%} over first {n} generated tokens"); print(f"First divergence: {first_difference if first_difference is not None else 'none'}"); print(f"Exact continuation: {exact}")
chat_result={"user_message":USER_MESSAGE,"system_message":SYSTEM_MESSAGE,"selected_candidate":selected,"selected_layers":record["layers"],"base_revision":base_revision,"prompt_tokens":int(prompt_ids.shape[1]),"max_new_tokens":int(MAX_NEW_TOKENS),"original_text":original_text,"cenn_text":cenn_text,"original_new_tokens":int(original_tokens.shape[1]),"cenn_new_tokens":int(cenn_tokens.shape[1]),"token_agreement":agreement,"first_divergence":first_difference,"exact_continuation":exact}
(OUT/"chat_comparison.json").write_text(json.dumps(chat_result,indent=2,ensure_ascii=False)); print("Saved:",OUT/"chat_comparison.json")
del cenn,original; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## Publish selected model + generated model card to Hugging Face


In [ ]:
HF_REPO = "vtava/Qwen3.5-0.8B-CeNN-Integrated-V1"
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN: raise RuntimeError("Add a Colab secret named HF_TOKEN with write permission.")
pub=[sys.executable,"-u",str(REPO/"scripts/package_qwen35_cenn_hf.py"),"--run-dir",str(OUT),"--repo-id",HF_REPO,"--token",HF_TOKEN]
subprocess.run(pub,cwd=REPO,check=True); print("✅ Published: https://huggingface.co/"+HF_REPO)
